# 2.5 MGO Price Forecasting with Ridge Regression

## Project Overview & Objectives
This notebook implements the machine learning forecasting pipeline for **Section 2.5 MGO Price Forecasting**.

**Core Objectives:**
1. **Near-Term MGO Movement Estimation:** Leverage historical Marine Gas Oil (`sg_mgo_themaritime`) price indications and selected market indicators (NYMEX Singapore Gasoil futures, Brent crude, Dubai crude, trade-weighted USD index, and bunker spreads) to estimate near-term MGO movements.
2. **Ridge Regularization (L2):** Handle multicollinearity across energy commodity series and prevent overfitting on sparse, irregular bunker quotes.
3. **Expected Direction & Range:** Output an expected direction (`UP`, `DOWN`, or `FLAT`) alongside prediction intervals (80%, 90%, 95% confidence bands).
4. **Uncertainty Quantification:** Quantify confidence levels based on model residual variance, market volatility, and MGO anchor freshness (elapsed quote staleness).
5. **Rigorous Validation:** Validate via expanding-window walk-forward cross validation (no data leakage) and benchmark against the **Persistence Baseline** and **Gasoil Transfer Baseline**.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.sans-serif"] = ["Segoe UI", "DejaVu Sans", "Arial"]

# Add src to sys.path
notebook_dir = Path.cwd()
src_dir = notebook_dir.parent / "src" if notebook_dir.name == "notebooks" else notebook_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config import TARGET_SERIES, MARKET_INDICATORS, BUNKER_INDICATORS
from data_loader import get_market_data
from feature_engineering import build_daily_market_features, align_mgo_targets_and_features
from ridge_model import RidgeMGOForecaster
from evaluate import run_walk_forward_evaluation
from predict import generate_forecast, format_forecast_report

print("Libraries and pipeline modules loaded successfully!")

## 1. Data Ingestion & Market Overview
We load canonical market observations (`market_observations.csv`) containing historical bunker prices and indicators.

In [ ]:
pivoted_df, audit_info = get_market_data()
print(f"Total raw rows: {audit_info['total_rows']:,}")
print(f"Target MGO observations: {audit_info['target_mgo_rows']} quotes from {audit_info['target_mgo_date_min']} to {audit_info['target_mgo_date_max']}")
print(f"Price range: ${audit_info['target_mgo_min_price']:.2f} to ${audit_info['target_mgo_max_price']:.2f} (Mean: ${audit_info['target_mgo_mean_price']:.2f})")

# Display key price series
key_series = [TARGET_SERIES] + MARKET_INDICATORS + BUNKER_INDICATORS
available_key = [s for s in key_series if s in pivoted_df.columns]
pivoted_df[available_key].dropna(subset=[TARGET_SERIES]).tail(10)

## 2. Market Indicator Correlation Analysis
Examining the correlation between Singapore MGO prices and market benchmarks.

In [ ]:
mgo_subset = pivoted_df[available_key].dropna(subset=[TARGET_SERIES])
corr_matrix = mgo_subset.corr()

plt.figure(figsize=(9, 7), dpi=120)
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", cbar_kws={'label': 'Correlation Coefficient'})
plt.title("Correlation Matrix: Singapore MGO vs Selected Market Indicators", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

## 3. Point-in-Time Feature Engineering
To strictly eliminate lookahead bias:
- Market indicators are strictly lagged ($t-1$).
- Indicator returns ($1$-day, $5$-day, $20$-day momentum) and crack spreads (`Gasoil - Brent`, `Brent - Dubai`) are derived.
- For each MGO quote, the elapsed days since the prior quote (`days_since_prev_quote`) and the Gasoil return over that interval are paired.

In [ ]:
features_df = build_daily_market_features(pivoted_df)
aligned_df, feature_cols = align_mgo_targets_and_features(pivoted_df, features_df)

print(f"Total engineered features: {len(feature_cols)}")
print(f"Clean aligned dataset rows: {len(aligned_df)}")
aligned_df[feature_cols[:8]].head()

## 4. Expanding-Window Walk-Forward Validation & Benchmark Comparison
We evaluate out-of-sample forecast accuracy using chronological expanding-window walk-forward validation (minimum 50 training observations, re-fitted sequentially).

We benchmark Ridge Regression against:
1. **Persistence Baseline:** Tomorrow's predicted price equals today's observed anchor price (random walk).
2. **Gasoil Transfer Baseline:** Assumes 1:1 percentage return transfer from Gasoil futures ($M_t = M_{t-1} \times (1 + r_G)$).

In [ ]:
val_df, eval_metrics = run_walk_forward_evaluation(aligned_df, feature_cols)

m_ridge = eval_metrics["models"]["ridge_regression"]
m_base = eval_metrics["models"]["persistence_baseline"]
m_gas = eval_metrics["models"]["gasoil_transfer_baseline"]

print(f"{'Model':<26} | {'MAE ($/MT)':<12} | {'RMSE ($/MT)':<12} | {'MAPE (%)':<10} | {'R2':<8}")
print("-" * 76)
print(f"{'Ridge Regression':<26} | {m_ridge['mae']:<12.2f} | {m_ridge['rmse']:<12.2f} | {m_ridge['mape_pct']:<10.2f} | {m_ridge['r2']:<8.4f}")
print(f"{'Persistence Baseline':<26} | {m_base['mae']:<12.2f} | {m_base['rmse']:<12.2f} | {m_base['mape_pct']:<10.2f} | {m_base['r2']:<8.4f}")
print(f"{'Gasoil Transfer Baseline':<26} | {m_gas['mae']:<12.2f} | {m_gas['rmse']:<12.2f} | {m_gas['mape_pct']:<10.2f} | {m_gas['r2']:<8.4f}")

print(f"\nDirectional Accuracy: {eval_metrics['directional_accuracy_pct']}%")
print(f"90% Prediction Interval Empirical Coverage: {eval_metrics['prediction_interval_coverage']['coverage_90pct']}% (Target: 90%)")
print(f"RMSE Error Reduction vs Persistence: {eval_metrics['error_reduction_vs_persistence']['rmse_reduction_pct']}%")

## 5. Forecast & Prediction Interval Visualizations
Plotting actual MGO quotes against out-of-sample Ridge forecasts and 80% / 90% prediction intervals.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6), dpi=140)

ax.plot(val_df.index, val_df["actual_price"], label="Actual Realized MGO", color="#1a252f", linewidth=1.8, marker="o", markersize=3, alpha=0.8)
ax.plot(val_df.index, val_df["pred_ridge_price"], label="Ridge Point Forecast", color="#e74c3c", linewidth=1.5)

ax.fill_between(val_df.index, val_df["lower_90"], val_df["upper_90"], color="#e74c3c", alpha=0.15, label="90% Prediction Interval")
ax.fill_between(val_df.index, val_df["lower_80"], val_df["upper_80"], color="#e74c3c", alpha=0.25, label="80% Prediction Interval")

ax.set_title("Singapore MGO Price Forecasting: Out-of-Sample Predictions & Uncertainty Bands", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Price (USD / Metric Tonne)", fontsize=11)
ax.legend(loc="upper left", frameon=True, facecolor="white", framealpha=0.9)
plt.tight_layout()
plt.show()

## 6. Feature Importance & Regularization Sensitivity
Inspect the standardized coefficients of the fitted Ridge model to understand which market indicators drive forecasted MGO returns.

In [ ]:
forecaster = RidgeMGOForecaster()
forecaster.fit(
    aligned_df[feature_cols],
    aligned_df["mgo_return"],
    aligned_df["mgo_lag_price"],
    aligned_df["mgo_price"]
)

coef_df = forecaster.get_feature_coefficients().head(12)
plt.figure(figsize=(10, 5), dpi=130)
colors = ["#2ecc71" if c > 0 else "#e74c3c" for c in coef_df["coefficient"]]

sns.barplot(data=coef_df, y="feature", x="coefficient", palette=colors)
plt.title(f"Top Standardized Feature Coefficients (Optimal Alpha = {forecaster.optimal_alpha_:.2f})", fontsize=13, fontweight="bold", pad=10)
plt.xlabel("Standardized Sensitivity to MGO Return", fontsize=11)
plt.ylabel("Market Indicator Feature", fontsize=11)
plt.axvline(0, color="#333333", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

## 7. Operational Inference & Advisory Output
Simulate an operational forecast for procurement decision-making communicating expected direction, forecast range, and uncertainty.

In [ ]:
latest_fc = generate_forecast(aligned_df, feature_cols, forecaster)
print(format_forecast_report(latest_fc))